# DubStudio · Seu estúdio no Colab

Dublagem com referência vocal autorizada e lip-sync opcional, lip-sync com áudio enviado e texto para voz. Os idiomas de um projeto são processados em sequência.
Na dublagem, lip-sync vem ativado por padrão. Se você o desativar no Studio, o notebook pula a análise facial e os modelos TalkNet/MuseTalk, preserva o vídeo inteiro e entrega **Dublagem sem lip-sync**; o relatório nunca afirma que a boca foi sincronizada nesse modo.

1. No Studio, escolha as mídias e clique em **Abrir no Colab**.
2. Aqui, selecione **Ambiente de execução → Alterar tipo de ambiente de execução → GPU**.
3. Clique em **Executar tudo** e selecione o ZIP baixado pelo Studio.
4. Importe o ZIP final em **Minhas criações** no Studio e confira a reprodução.

A GPU gratuita e a duração da sessão dependem do Colab. Este notebook não abre túnel, não automatiza login e não tenta evitar o encerramento. Confira a [FAQ do Colab](https://research.google.com/colaboratory/faq.html): runtimes gerenciados proíbem deepfakes. O notebook não afirma que lip-sync facial é autorizado pelo Google e registra qualquer bloqueio/falha sem declarar a etapa concluída.

In [ ]:
# @title 1 · Conferir GPU antes de instalar ou baixar modelos
import os, sys, shutil, subprocess, json, zipfile, uuid
from pathlib import Path, PurePosixPath

if not shutil.which('nvidia-smi'):
    raise RuntimeError('GPU indisponível. Selecione GPU no menu Ambiente de execução e reconecte; nenhum modelo foi baixado.')
gpu = subprocess.run(['nvidia-smi', '--query-gpu=name,memory.total,memory.free', '--format=csv,noheader,nounits'], check=True, capture_output=True, text=True)
if not gpu.stdout.strip():
    raise RuntimeError('O Colab não concedeu GPU nesta sessão. Tente novamente quando o serviço disponibilizar.')
print('GPU: ' + gpu.stdout.strip())
print(f'Espaço livre no runtime: {shutil.disk_usage("/content").free / 1024**3:.1f} GB')

In [ ]:
# @title 2 · Enviar pacote do Studio (ou pacote com revisão aprovada)
from google.colab import files
uploaded = files.upload()
if len(uploaded) != 1:
    raise RuntimeError('Selecione exatamente um ZIP preparado pelo DubStudio.')
package_name = next(iter(uploaded))
if not zipfile.is_zipfile(package_name):
    raise RuntimeError('O arquivo selecionado não é um pacote ZIP válido.')
package_root = Path('/content/dubstudio-packages') / uuid.uuid4().hex
package_root.mkdir(parents=True, exist_ok=True)
with zipfile.ZipFile(package_name) as archive:
    infos = archive.infolist()
    if len(infos) > 10000 or sum(item.file_size for item in infos) > 15 * 1024**3:
        raise RuntimeError('Pacote excede o limite de tamanho do notebook.')
    names = set()
    for item in infos:
        path = PurePosixPath(item.filename.replace(chr(92), '/'))
        if path.is_absolute() or '..' in path.parts or ':' in str(path) or item.filename in names or (item.external_attr >> 16) & 0o170000 == 0o120000:
            raise RuntimeError('Pacote contém caminho inseguro, arquivo repetido ou link simbólico.')
        names.add(item.filename)
    archive.extractall(package_root)
required = ['dubstudio_bootstrap.py', 'dubstudio_runner_v2.py', 'dubstudio_colab_worker.py', 'dubstudio_translate.py', 'requirements-app.txt', 'requirements-musetalk.txt']
if any(not (package_root / 'worker' / name).is_file() for name in required):
    raise RuntimeError('Pacote antigo/incompleto. Exporte novamente pelo Studio atualizado.')
sys.path.insert(0, str(package_root / 'worker'))
import dubstudio_runner_v2 as runner
manifest = runner.validate_manifest(package_root)
print(f'Projeto {manifest["job_id"]} · {manifest["operation"]} · {", ".join(manifest["target_languages"])}')
print('Mídias e checksums conferidos.')
del uploaded

In [ ]:
# @title 3 · Cache e retomada
armazenamento = "Usar escolha do Studio" # @param ["Usar escolha do Studio", "Salvar no Drive (autorizar)", "Somente esta sessão"]
usar_drive = bool(manifest.get('options', {}).get('cache_on_drive', True))
if armazenamento == 'Salvar no Drive (autorizar)':
    usar_drive = True
elif armazenamento == 'Somente esta sessão':
    usar_drive = False
runtime_root = Path('/content/dubstudio')
if usar_drive:
    print('Cache no Drive selecionado. Autorize sua própria conta na caixa do Google para continuar; mídias e checkpoints não serão publicados.')
    from google.colab import drive
    drive.mount('/content/drive')
    storage_root = Path('/content/drive/MyDrive/DubStudio')
    print('Pesos e checkpoints salvos no Drive; o processamento usa o disco local /content.')
else:
    storage_root = runtime_root / 'session-storage'
    print('Cache somente nesta sessão: uma desconexão definitiva poderá exigir reiniciar o trabalho.')
storage_root.mkdir(parents=True, exist_ok=True)

In [ ]:
# @title 4 · Preparar o ambiente uma vez nesta sessão
import dubstudio_bootstrap as bootstrap
needs_lipsync = runner.needs_lipsync(manifest)
bootstrap.prepare(runtime_root, storage_root, needs_lipsync)
worker_env = bootstrap.environment(runtime_root)
worker_env['DUBSTUDIO_RUNTIME_ROOT'] = str(runtime_root)
worker_env['DUBSTUDIO_STORAGE_ROOT'] = str(storage_root)

In [ ]:
# @title 5 · Processar ou retomar e baixar o resultado
result = Path(f'/content/DubStudio-{manifest["job_id"]}-resultado.zip')
result.unlink(missing_ok=True)  # Um pacote antigo nunca representa esta nova tentativa.
command = [str(runtime_root / 'app-env/bin/python'), '-u', str(package_root / 'worker/dubstudio_runner_v2.py'), '--package-root', str(package_root), '--work-root', str(runtime_root / 'jobs'), '--storage-root', str(storage_root), '--output', str(result)]
process = subprocess.Popen(command, env=worker_env, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, encoding='utf-8', errors='replace', bufsize=1)
for line in process.stdout:
    print(line, end='', flush=True)
returncode = process.wait()
if result.exists() and zipfile.is_zipfile(result):
    with zipfile.ZipFile(result) as archive:
        if 'review.json' in archive.namelist():
            print('TRADUÇÃO PARA REVISAR: importe este ZIP no Studio, aprove e execute novamente com o pacote aprovado.')
        else:
            result_info = json.loads(archive.read('result.json'))
            complete = sum(item['status'] == 'complete' for item in result_info['outputs'])
            print(f'{complete}/{len(result_info["outputs"])} idioma(s) exportado(s). Importe no Studio e confira tradução, timbre e lip-sync na reprodução.')
    files.download(str(result))
else:
    raise RuntimeError('A execução terminou sem pacote. O log acima indica a etapa exata; checkpoints concluídos estão preservados para retomada.')
if returncode:
    print('Houve falha; o pacote inclui diagnóstico. Reenvie o mesmo projeto após corrigir a causa para retomar.')

## Próximo vídeo e retomada

Para outro vídeo, execute a partir da célula **2 · Enviar pacote**. O ambiente preparado e os pesos íntegros serão reutilizados.

Após uma desconexão, conecte a GPU e execute tudo com o mesmo pacote e Drive. As etapas com checksums conferidos serão reaproveitadas. Ambientes Python são preparados no runtime novo; executáveis antigos não são copiados do Drive.

No Colab o progresso corresponde às etapas e contadores realmente executados. O Studio local não recebe a porcentagem desta sessão: durante a execução, acompanhe este notebook. O relatório informa GPU, uso de cache, tempo por etapa e qualquer falha. Nunca aceite apenas a mensagem de término como prova visual de qualidade.